# Screening Sentinel-2 after the archive, March 2025 to October 2026

Same window and clear-sky rule as notebook 02, from March 2025 onward. For every date it records cloud, snow and open water within 1.5 km of the lake (SCL water class), to check whether the lake refilled and drained after the archive ends, and it saves true-colour thumbnails of every clear date from April to October.

**Output** (`Shisper_S2_screening`): `scene_screening_2025_2026.csv`, `lake_water_2025_2026.png` and `clear_thumbnails_2025_2026.png`. CPU runtime, about 5 minutes.

## Setup

In [ ]:
%pip install -q pystac-client planetary-computer rasterio

import os, json, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import pystac_client, planetary_computer, rasterio
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = '/content/drive/MyDrive'
SCENE_DIR = f'{DRIVE}/Shisper_lake_S2'
OUT = f'{DRIVE}/Shisper_S2_screening'
os.makedirs(OUT, exist_ok=True)

LAKE = (36.36061, 74.56276)
if os.path.exists(f'{SCENE_DIR}/lake_location.json'):
    j = json.load(open(f'{SCENE_DIR}/lake_location.json')); LAKE = (j['lat'], j['lon'])
KM, NEAR_KM = 5.0, 1.5
START, END = '2025-03-01', '2026-10-31'
MAX_CLOUD, MIN_VALID = 0.10, 0.95
CLOUD = [3, 8, 9, 10]                             # SCL: cloud shadow, cloud medium and high probability, thin cirrus
dlat = KM / 2 / 111.32
dlon = dlat / np.cos(np.radians(LAKE[0]))
BOX = (LAKE[1] - dlon, LAKE[0] - dlat, LAKE[1] + dlon, LAKE[0] + dlat)
catalog = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',
                                    modifier=planetary_computer.sign_inplace)

## Cloud, snow and lake water on every date

In [ ]:
items = []
for y0 in range(int(START[:4]), int(END[:4]) + 1):
    a, b = max(START, f'{y0}-01-01'), min(END, f'{y0}-12-31')
    for k in range(4):
        try:
            items += list(catalog.search(collections=['sentinel-2-l2a'], bbox=BOX, datetime=f'{a}/{b}', limit=1000).items())
            break
        except Exception:
            if k == 3:
                raise
            time.sleep(10 * (k + 1))
print(len(items), 'products cover the window')

NPX = int(round(KM * 1000 / 20))                  # SCL is at 20 m
yy, xx = np.mgrid[0:NPX, 0:NPX]
NEAR = np.hypot(yy - (NPX - 1) / 2, xx - (NPX - 1) / 2) * (KM / NPX) <= NEAR_KM

def read_window(it, key, shape, bands=1):
    href = it.assets[key].href
    for k in range(4):
        try:
            with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', GDAL_HTTP_MAX_RETRY='5'):
                with rasterio.open(href) as src:
                    w = from_bounds(*transform_bounds('EPSG:4326', src.crs, *BOX), transform=src.transform)
                    idx = 1 if bands == 1 else list(range(1, bands + 1))
                    return src.read(idx, window=w, out_shape=shape, boundless=True, fill_value=0,
                                    resampling=Resampling.nearest)
        except Exception:
            if k == 3:
                return None
            time.sleep(3 * (k + 1))

def day(it):
    return it.datetime.strftime('%Y-%m-%d')

failed, MOS = 0, {}
with ThreadPoolExecutor(16) as pool:
    for it, a in zip(items, pool.map(lambda it: read_window(it, 'SCL', (NPX, NPX)), items)):
        if a is None:
            failed += 1
            continue
        d = day(it)
        MOS[d] = a if d not in MOS else np.where(MOS[d] == 0, a, MOS[d])
print(f'{len(items) - failed} windows read, {failed} failed')

rows = []
for d, m in sorted(MOS.items()):
    valid = m > 0
    nv = max(int(valid.sum()), 1)
    rows.append(dict(date=d, valid=round(valid.mean(), 3), cloud=round(float(np.isin(m, CLOUD).sum()) / nv, 3),
                     snow=round(float((m == 11).sum()) / nv, 3),
                     water_km2=round(float(((m == 6) & NEAR).sum()) * 0.02 * 0.02, 3)))   # SCL class 6: water
T = pd.DataFrame(rows)
T['clear'] = (T.valid >= MIN_VALID) & (T.cloud <= MAX_CLOUD)
T['month'] = T.date.str[5:7].astype(int)
T.to_csv(f'{OUT}/scene_screening_2025_2026.csv', index=False)

## Lake water through time and thumbnails of the clear dates

In [ ]:
d = pd.to_datetime(T.date)
fig, ax = plt.subplots(figsize=(14, 3.6))
ax.plot(d[~T.clear], T.water_km2[~T.clear], 'o', ms=4, mfc='none', c='0.6', label='cloudy date')
ax.plot(d[T.clear], T.water_km2[T.clear], 'o', ms=5, c='#2c7bb6', label='clear date')
ax.set_ylabel(f'open water within {NEAR_KM} km\nof the lake (km²)')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout(); plt.savefig(f'{OUT}/lake_water_2025_2026.png', dpi=110); plt.show()

C = T[T.clear & T.month.between(4, 10)].reset_index(drop=True)
by_day, CD = {}, set(C.date)
for it in items:
    if day(it) in CD and 'visual' in it.assets:
        by_day.setdefault(day(it), []).append(it)

def thumb(d):
    out = None
    for it in by_day.get(d, []):
        a = read_window(it, 'visual', (3, 200, 200), bands=3)
        if a is not None:
            out = a if out is None else np.where(out == 0, a, out)
    return d, out

with ThreadPoolExecutor(8) as pool:
    TH = dict(pool.map(thumb, list(C.date)))
cols = 6
rws = max(1, int(np.ceil(len(C) / cols)))
fig, ax = plt.subplots(rws, cols, figsize=(3 * cols, 3.6 * rws), squeeze=False)
for k in range(rws * cols):
    a = ax[k // cols, k % cols]
    a.set_xticks([]); a.set_yticks([])
    if k >= len(C):
        a.axis('off')
        continue
    r = C.iloc[k]
    if TH.get(r.date) is not None:
        a.imshow(np.transpose(TH[r.date], (1, 2, 0)))
    a.set_title(f'{r.date}\nwater {r.water_km2:.3f} km², cloud {r.cloud:.0%}', fontsize=9)
plt.tight_layout(h_pad=2.5); plt.savefig(f'{OUT}/clear_thumbnails_2025_2026.png', dpi=90); plt.show()

print(f'dates with data {START} to {END}: {len(T)} | clear: {int(T.clear.sum())} | clear April to October: {len(C)}')
for y0 in sorted(set(T.date.str[:4])):
    Y = T[T.date.str.startswith(y0)]
    print(f'  {y0}: {len(Y)} dates, {int(Y.clear.sum())} clear, largest water area on a clear date '
          f'{Y.water_km2[Y.clear].max() if Y.clear.any() else float("nan"):.3f} km²')